In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from typing import Optional
from pydantic import BaseModel, Field
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from difflib import SequenceMatcher
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from pathlib import Path

PROJECT_ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "ebm_nlp_2_00").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find ebm_nlp_2_00 from the current notebook directory")

DATA_FILE = PROJECT_ROOT / "ebm_nlp_2_00" / "processed" / "ebm_abstracts_full.npz"
OUTPUT_DIR = PROJECT_ROOT / "task2-llm"
OUTPUT_DIR.mkdir(exist_ok=True)
PREDICTION_COLUMNS = ["method", "doc_id", "text", "Pop_gold", "Pop_pred", "Int_gold", "Int_pred", "Out_gold", "Out_pred"]

class PIOExtraction(BaseModel):
    population: Optional[str] = Field(description="The patients or problem. Return null if none.")
    intervention: Optional[str] = Field(description="The main treatment. Return null if none.")
    outcome: Optional[str] = Field(description="The primary results. Return null if none.")

llm = OllamaLLM(model="llama3.1", format="json", temperature=0)
parser = JsonOutputParser(pydantic_object=PIOExtraction)

def extract_text_from_mask(text, mask):
    words = text.split()
    extracted_spans = []
    current_span = []
    
    for word, label in zip(words, mask):
        if label == 1:
            current_span.append(word)
        else:
            if current_span:
                extracted_spans.append(" ".join(current_span))
                current_span = []
                
    if current_span: 
        extracted_spans.append(" ".join(current_span))
        
    if not extracted_spans:
        return "null"
        
    return " ; ".join(extracted_spans)

def evaluate_extraction(original_text, extracted_text, gt_mask, threshold=0.6):
    if not extracted_text or str(extracted_text).lower() in ('null', 'none'):
        return "None", None 
        
    items = [item.strip() for item in str(extracted_text).split(';')]
    
    orig_words = original_text.split()
    item_scores = []
    
    for item in items:
        if not item: continue
        
        ext_words = item.split()
        window = len(ext_words)
        best_ratio, best_start = 0, 0
        
        for i in range(len(orig_words) - window + 1):
            window_text = " ".join(orig_words[i:i+window])
            ratio = SequenceMatcher(None, item.lower(), window_text.lower()).ratio()
            if ratio > best_ratio:
                best_ratio, best_start = ratio, i

        if best_ratio < threshold:
            item_scores.append(0.0) # (Hallucination)
        else:
            mask_slice = gt_mask[best_start : best_start + window]
            if any(label == 1 for label in mask_slice):
                item_scores.append(1.0) # (Hit)
            else:
                item_scores.append(0.0) # (Miss)

    if not item_scores:
        return "None", None

    avg_precision = sum(item_scores) / len(item_scores)
    
    if avg_precision == 1.0:
        return "Hit (Relaxed)", 1.0
    elif avg_precision == 0.0:
        return "Hallucinated", 0.0
    else:
        return "Partial Hit", avg_precision

def run_dynamic_experiment(num_shots, train_texts, train_masks, test_texts, test_masks, test_ids, embedder, train_embeddings):
    print(f"\n" + "="*60)
    print(f"RUNNING {num_shots}-SHOT DYNAMIC RAG EXPERIMENT")
    print("="*60)

    template = """You are a medical researcher. Extract PIO elements in JSON format.
    CRITICAL RULE: Your extractions MUST be exact, continuous substrings copied directly from the abstract. Do not add or change any words.
    
    {format_instructions}
    
    {few_shot_examples}
    
    --- ACTUAL TASK ---
    Abstract: {abstract}
    Output:"""

    pipeline = PromptTemplate(
        template=template,
        input_variables=["abstract", "few_shot_examples"], 
        partial_variables={
            "format_instructions": parser.get_format_instructions()
        }
    ) | llm | parser

    limit = min(1000, len(test_texts))
    results = []
    
    for i in tqdm(range(limit), desc=f"Processing Abstracts ({num_shots}-shot Dynamic)"):
        text = test_texts[i]
        
        try:
            test_embedding = embedder.encode([text])
            
            similarities = cosine_similarity(test_embedding, train_embeddings)[0]
            
            top_k_indices = np.argsort(similarities)[-num_shots:][::-1]
            
            dynamic_few_shot_string = ""
            for idx in top_k_indices:
                train_text = train_texts[idx]
                pop_str = extract_text_from_mask(train_text, train_masks['Pop'][idx])
                int_str = extract_text_from_mask(train_text, train_masks['Int'][idx])
                out_str = extract_text_from_mask(train_text, train_masks['Out'][idx])
                
                dynamic_few_shot_string += f"--- EXAMPLE ---\n"
                dynamic_few_shot_string += f"Abstract: {train_text}\n"
                dynamic_few_shot_string += f"Output: {{\\\"population\\\": \\\"{pop_str}\\\", \\\"intervention\\\": \\\"{int_str}\\\", \\\"outcome\\\": \\\"{out_str}\\\"}}\n\n"

            res = pipeline.invoke({
                "abstract": text,
                "few_shot_examples": dynamic_few_shot_string.strip()
            })
            
            row = {
                "method": f"llm_dynamic_{num_shots}_shot",
                "doc_id": test_ids[i],
                "text": text,
                "Pop_gold": extract_text_from_mask(text, test_masks["Pop"][i]),
                "Int_gold": extract_text_from_mask(text, test_masks["Int"][i]),
                "Out_gold": extract_text_from_mask(text, test_masks["Out"][i]),
            }
            
            for short_key, full_key in [('Pop', 'population'), ('Int', 'intervention'), ('Out', 'outcome')]:
                extracted = res.get(full_key)
                row[f"{short_key}_pred"] = extracted if extracted else "null"
                
            results.append(row)
                
        except Exception as e:
            print(f"\nError processing sample {i}: {e}") 

    if results:
        df = pd.DataFrame(results)
        filename = f"llm_dynamic_{num_shots}_shot_predictions.csv"
        df = df.reindex(columns=PREDICTION_COLUMNS)
        df.to_csv(OUTPUT_DIR / filename, index=False)
        
        print(f"\n{num_shots} SHOT DYNAMIC - TRUE RELAXED PRECISION SCORES")
        print("-" * 50)
        for key in ['Pop', 'Int', 'Out']:
            coverage = (df[f'{key}_pred'].fillna('null').str.lower() != 'null').mean()
            print(f"{key:<12} Coverage : {coverage * 100:.1f}%")
        print("-" * 50)


print("Loading dataset...")
data = np.load(DATA_FILE, allow_pickle=True)
    
train_texts = data['train_texts']
train_masks = {'Pop': data['train_p'], 'Int': data['train_i'], 'Out': data['train_o']}
    
test_texts = data['test_texts']
test_masks = {'Pop': data['test_p'], 'Int': data['test_i'], 'Out': data['test_o']}
test_ids = list(data['test_ids']) if 'test_ids' in data.files else [f"test_{i}" for i in range(len(test_texts))]

print("Loading Sentence Transformer model (this will take a moment the first time)...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
    
print("Pre-computing embeddings for all training texts (this saves time during the loop)...")
train_embeddings = embedder.encode(train_texts[:200])
print("Embeddings ready!")

# Test 1, 2, and 3 shots dynamically
shots_to_test = [1, 2, 3, 4, 5]
    
for num_shots in shots_to_test:
    run_dynamic_experiment(
        num_shots, 
        train_texts, 
        train_masks, 
        test_texts, 
        test_masks, 
        test_ids, 
        embedder, 
        train_embeddings
    )

Loading dataset...
Loading Sentence Transformer model (this will take a moment the first time)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pre-computing embeddings for all training texts (this saves time during the loop)...
Embeddings ready!

RUNNING 1-SHOT DYNAMIC RAG EXPERIMENT


Processing Abstracts (1-shot Dynamic): 100%|██████████| 184/184 [11:11<00:00,  3.65s/it]


...


3 SHOT DYNAMIC - TRUE RELAXED PRECISION SCORES
--------------------------------------------------
Pop          Coverage : 99.5%
Int          Coverage : 100.0%
Out          Coverage : 100.0%
--------------------------------------------------

RUNNING 4-SHOT DYNAMIC RAG EXPERIMENT


Processing Abstracts (4-shot Dynamic): 100%|██████████| 184/184 [10:31<00:00,  3.43s/it]



4 SHOT DYNAMIC - TRUE RELAXED PRECISION SCORES
--------------------------------------------------
Pop          Coverage : 98.4%
Int          Coverage : 99.5%
Out          Coverage : 100.0%
--------------------------------------------------

RUNNING 5-SHOT DYNAMIC RAG EXPERIMENT


Processing Abstracts (5-shot Dynamic): 100%|██████████| 184/184 [17:52<00:00,  5.83s/it]  


5 SHOT DYNAMIC - TRUE RELAXED PRECISION SCORES
--------------------------------------------------
Pop          Coverage : 99.5%
Int          Coverage : 100.0%
Out          Coverage : 100.0%
--------------------------------------------------


In [2]:
import pandas as pd

def calculate_metrics(file_paths):
    all_summaries = []
    
    for path in file_paths:
        try:
            df = pd.read_csv(OUTPUT_DIR / path)
            
            summary = {"Experiment": df['method'].iloc[0] if 'method' in df else path}
            
            for pio in ['Pop', 'Int', 'Out']:
                pred = df[f'{pio}_pred'].fillna('null').astype(str).str.lower()
                gold = df[f'{pio}_gold'].fillna('null').astype(str).str.lower()
                summary[f"{pio}_Coverage"] = f"{(pred != 'null').mean()*100:.1f}%"
                summary[f"{pio}_Gold_Coverage"] = f"{(gold != 'null').mean()*100:.1f}%"
                
            all_summaries.append(summary)
        except Exception as e:
            print(f"Error processing {path}: {e}")
            
    return pd.DataFrame(all_summaries)

my_files = [
    "llm_dynamic_1_shot_predictions.csv", 
    "llm_dynamic_2_shot_predictions.csv", 
    "llm_dynamic_3_shot_predictions.csv",
    "llm_dynamic_4_shot_predictions.csv",
    "llm_dynamic_5_shot_predictions.csv",
    
]

report_table = calculate_metrics(my_files)
print(report_table.to_string(index=False))

        Experiment Pop_Coverage Pop_Gold_Coverage Int_Coverage Int_Gold_Coverage Out_Coverage Out_Gold_Coverage
llm_dynamic_1_shot        96.2%            100.0%       100.0%             98.9%       100.0%             97.8%
llm_dynamic_2_shot        96.2%            100.0%        98.9%             98.9%        99.5%             97.8%
llm_dynamic_3_shot        99.5%            100.0%       100.0%             98.9%       100.0%             97.8%
llm_dynamic_4_shot        98.4%            100.0%        99.5%             98.9%       100.0%             97.8%
llm_dynamic_5_shot        99.5%            100.0%       100.0%             98.9%       100.0%             97.8%
